<a href="https://colab.research.google.com/github/dr-bankert-augustana/PHYS_200/blob/main/Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a name="Notebook-Start"></a>

---

<font size = 7> <b> Project 2 (Data Analysis) </b> </font>

---

In [ ]:
##=============================================================================================##
## Import Libraries:                                                                           ##
##=============================================================================================##

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display_html

import tensorflow as tf
from tensorflow import keras
from sklearn.linear_model import LinearRegression
from tensorflow.keras import layers
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
#@title This cell defines some helpful functions.

##=============================================================================================##
## Function:  display_dataframes                                                               ##
##                                                                                             ##
## Purpose:   Display multiple DataFrames side-by-side with titles                             ##
##                                                                                             ##
## Input(s):  dataframe_list - List of DataFrames to be displayed                              ##
##            title_list     - List of titles for the displayed DataFrames                     ##
##            n_items        - Number of items to display (optional, default = 5)              ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def display_dataframes(dataframe_list, title_list, n_items = 5, tail = False):

  ##===========================================================================================##
  ## Create a String for Housing the Commands to Be Sent to the display_html() Function:       ##
  ##===========================================================================================##

  html_str = ""

  ##===========================================================================================##
  ## Loop Over the Elements in the item_list and title_list:                                   ##
  ##===========================================================================================##

  for df, title in zip(dataframe_list, title_list):

    # Convert the current dataframe info to html:

    if (tail == True):

      html_df = pd.DataFrame(df).tail(n_items).to_html()

    else:

      html_df = pd.DataFrame(df).head(n_items).to_html()

    ##=========================================================================================##
    ## Wrap title and DataFrames in a Styled HTML <div>:                                       ##
    ##=========================================================================================##

    html_str += "<div style='display: inline-block; margin-right: 20px; vertical-align: top;'>"

    html_str += "<h3 style='text-align: center;'>" + str(title) + "</h3><hr>" + str(html_df)

    html_str += "</div>"

  ##===========================================================================================##
  ## Send the HTML String to the display_html() Function:                                      ##
  ##===========================================================================================##

  display_html(html_str, raw = True)

##=============================================================================================##
## Function:  create_model                                                                     ##
##                                                                                             ##
## Purpose:   Display models' parameters and loss in a DataFrame                               ##
##                                                                                             ##
## Input(s):  model_list - List of models' names to be displayed                               ##
##                                                                                             ##
## Output(s): None                                                                             ##
##=============================================================================================##

def create_model(X, y, name, bias_on = True):

  ##===========================================================================================##
  ## Use a LinearRegression Object to Find the Best Fit for the Model:                         ##
  ##===========================================================================================##

  # Create a LinearRegression object with a forced-origin intercept:

  model = LinearRegression(fit_intercept = bias_on)

  # Fit the LinearRegression objects to the features and target:

  model.fit(X, y)

  # Get the coefficient and bias for the model:

  coefs = np.round(model.coef_[0], 2)
  bias  = np.round(model.intercept_, 2)

  ##===========================================================================================##
  ## Get The Model Predictions:                                                                ##
  ##===========================================================================================##

  predictions = model.predict(X)

  ##===========================================================================================##
  ## Calculate Model Losses:                                                                   ##
  ##===========================================================================================##

  loss = np.sqrt(mean_squared_error(y, predictions))

  ##===========================================================================================##
  ## Create a DataFrame to Store the Model's Results and Predictions:                          ##
  ##===========================================================================================##

  # Create the results DataFrame:

  results_df = pd.DataFrame({"Name": name, "Coefs": [coefs], "Bias": bias, "Loss": loss})

  # Create the predictions DataFrame:

  predictions_df = pd.DataFrame({"Target": X.iloc[:, 0]})

  predictions_df["Predictions"] = predictions

  ##===========================================================================================##
  ## Return the Model's Results and Predictions:                                               ##
  ##===========================================================================================##

  return results_df, predictions_df

##=============================================================================================##
## Function:  plot_data                                                                        ##
##                                                                                             ##
## Purpose:   Create a scatterplot with optional model overlays and error bands                ##
##                                                                                             ##
## Input(s):  x_data        - List of data points' x-axis values                               ##
##            y_data        - List of data points' y-axis values                               ##
##            title         - Graph title                                                      ##
##            axis_labels   - Override axis labels [x_label, y_label] (optional)               ##
##            model_list    - List of model predictions to overlay (optional)                  ##
##            color_list    - Colors for each model line (optional)                            ##
##            label_list    - Labels for each model line (optional)                            ##
##            error_display - Show +/- error band around first model (default is False)        ##
##            error         - Error value for shaded band (optional)                           ##
##                                                                                             ##
## Output(s): graph       - Matplotlib axes object, can be used for overplotting               ##
##=============================================================================================##

def plot_data(x_data, y_data, title, axis_labels = [], model_list = [], color_list = [],
              label_list = [], error_display = False, error = 0):

  ##===========================================================================================##
  ## Setup the Graph:                                                                          ##
  ##===========================================================================================##

  # Create the Matplotlib figure:

  figure = plt.figure(figsize = (12, 9))

  # Add a graph to the figure:

  graph = figure.add_subplot()

  # Set the graph background Color:

  graph.set_facecolor('lightcyan')

  # Set the graph title:

  graph.set_title(title, fontsize = 20)

  # Set the x_label and y_label:

  if (axis_labels != []):

    graph.set_xlabel(axis_labels[0], fontsize = 14)

    graph.set_ylabel(axis_labels[1], fontsize = 14)

  # Apply a grid to the graph:

  graph.grid(which = 'both')

  # Adjust the x-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'x', tight = False)

  # Adjust the y-axis scale of the graph:

  graph.autoscale(enable = True, axis = 'y', tight = False)

  ##===========================================================================================##
  ## Add Data to the Graph:                                                                    ##
  ##===========================================================================================##

  # Create a scatterplot of the data:

  sns.scatterplot(x = x_data, y = y_data, ax = graph)

  # Overlay model predictions:

  for i in range(0, len(model_list)):

    graph.plot(model_list[i]["Feature"], model_list[i]["Predictions"], color = color_list[i],
               label = label_list[i])

  ##===========================================================================================##
  ## If requested, show the +/- error bounds:                                                  ##
  ##===========================================================================================##

  if ((error_display == True) and model_list != []):

    # Create the error+ model:

    model_plus_error  = model_list[-1]["Predictions"] + error

    # Create the error- model:

    model_minus_error = model_list[-1]["Predictions"] - error

    # Store the error+ and error- models in a DataFrame and Sort by x_data values:

    error_df = pd.DataFrame({
        'X-Data': model_list[0]["Feature"],
        'Model+Error': model_plus_error,
        'Model-Error': model_minus_error
    }).sort_values(by = "X-Data")

    # Use graph.fill to highlight the region between the error+ and error- models:

    graph.fill_between(error_df['X-Data'], error_df['Model+Error'], error_df['Model-Error'],
                       alpha = 0.5, color = (0.6, 0.6, 0.6), label = "Error Bounds")

  ##===========================================================================================##
  ## Apply the Legend and Return the graph Object:                                             ##
  ##===========================================================================================##

  # Add the graph legend:

  if (label_list != []): graph.legend()

  # Return the graph:

  return graph

<a name="Learning-Outcomes"></a>

---

#<font size = 6> <b> 1. Generate the Data </b> </font>

---

In [ ]:
##=============================================================================================##
## Generate the Angular Positions and Time Data:                                               ##
##=============================================================================================##

# Local Acceleration Due to Gravity (Meters Per Second Squared):

g = 9.8

# Pendulum Length (Meters):

L = 1

# Theta Max (Radians):

theta_max = 0.1

# Sample Time Values (Seconds):

t = np.linspace(0, 10, 500)

# Phase Speed:

omega_p = np.sqrt(g / L)

# Theta (Radians):

theta = theta_max * np.cos(omega_p * t)

##=============================================================================================##
## Add Noise to Data:                                                                          ##
##=============================================================================================##

theta_noisy = theta + np.random.normal(0, 0.01, size = theta.shape)

##=============================================================================================##
## Identify Features and Targets:                                                              ##
##=============================================================================================##

# Set the Feature Data:

X = pd.DataFrame({"Time (s)": t})

# Set the Target Data:

y = pd.DataFrame({"Angle (rad)": theta_noisy})

# Display the Data:

display_dataframes([X, y], ["Feature Data", "Target Data"])

In [ ]:
##=============================================================================================##
## Split the Data Into Testing and Trainging Subsets (Random State = 42):                      ##
##=============================================================================================##

## YOUR CODE HERE

<a name="Learning-Outcomes"></a>

---

#<font size = 6> <b> 1. Visualize the Data </b> </font>

---

In [ ]:
##=============================================================================================##
## Create a Plot of the Data:                                                                  ##
##=============================================================================================##

# Set the title of the plot:

title_1 = "Angular Position vs Time (Training Data)"
title_2 = "Angular Position vs Time (Testing Data)"

# Set the x-axis and y-axis labels:

x_label = "Time (s)"
y_label = "Angle (rad)"

# Create a scatterplot of the feature data vs the target data:

graph_1 = plot_data(X_train[x_label], y_train[y_label], title_1, [x_label, y_label])

graph_2 = plot_data(X_test[x_label], y_test[y_label], title_2, [x_label, y_label])


<a name="Learning-Outcomes"></a>

---

#<font size = 6> <b> 2. Create A Simple Linear Model </b> </font>

---

In [ ]:
##=============================================================================================##
## Use a LinearRegression Object to Find the Best Fit for the Model:                           ##
##=============================================================================================##

# Create a LinearRegression object:

model_simple = ## YOUR CODE HERE

# Fit the LinearRegression objects to the features and target:

## YOUR CODE HERE

# Get the coefficient and bias for the model:

coefs = ## YOUR CODE HERE
bias = ## YOUR CODE HERE

##=============================================================================================##
## Get The Model Predictions:                                                                  ##
##=============================================================================================##

predictions = ## YOUR CODE HERE

##=============================================================================================##
## Calculate Model Losses:                                                                     ##
##=============================================================================================##

loss =  ## YOUR CODE HERE

print("Linear Model Loss: ", np.round(loss, 5), "rad")

##=============================================================================================##
## Create a DataFrame to Store the Model's Results and Predictions:                            ##
##=============================================================================================##

# Create the results DataFrame:

results_df = pd.DataFrame({"Name": "Linear Regression Model", "Coefs": [coefs], "Bias": bias, "Loss": loss})

# Create the predictions DataFrame:

predictions_df = pd.DataFrame({"Feature": X_test.iloc[:, 0]})

predictions_df["Predictions"] = predictions

predictions_df["Loss"] = loss

##=============================================================================================##
## Create a Plot of the Data:                                                                  ##
##=============================================================================================##

# Set the title of the plot:

title = "Angular Position vs Time"

# Set the x-axis and y-axis labels:

x_label = "Time (s)"
y_label = "Angle (rad)"

# Create the model, color, and label lists:

model_list = [predictions_df]
color_list = ["red"]
label_list = ["Linear Model"]

# Create a scatterplot of the feature data vs the target data:

graph = plot_data(X[x_label], y[y_label], title, [x_label, y_label], model_list, color_list,
                  label_list, error_display = True, error = predictions_df['Loss'])

<a name="Learning-Outcomes"></a>

---

#<font size = 6> <b> 3. Create A Linearized Model </b> </font>

---

In [ ]:
##=============================================================================================##
## Use a LinearRegression Object to Find the Best Fit for the Model:                           ##
##=============================================================================================##

# Linearize the Testing and Training Feature Data:

X_linear_train = ## YOUR CODE HERE

X_linear_test = ## YOUR CODE HERE

# Create a LinearRegression object:

## YOUR CODE HERE

# Fit the LinearRegression objects to the features and target:

## YOUR CODE HERE

# Get the coefficient and bias for the model:

## YOUR CODE HERE

##=============================================================================================##
## Get The Model Predictions:                                                                  ##
##=============================================================================================##

## YOUR CODE HERE

##=============================================================================================##
## Calculate Model Losses:                                                                     ##
##=============================================================================================##

## YOUR CODE HERE

print("Linearized Model Loss: ", np.round(loss, 5), "rad")

##=============================================================================================##
## Create a DataFrame to Store the Model's Results and Predictions:                            ##
##=============================================================================================##

# Create the results DataFrame:

results_df = pd.DataFrame({"Name": "Linear Regression Model", "Coefs": [coefs], "Bias": bias, "Loss": loss})

# Create the predictions DataFrame:

predictions_df = pd.DataFrame({"Feature": X_test.iloc[:, 0]})

predictions_df["Predictions"] = predictions

predictions_df["Loss"] = loss

predictions_df.sort_index(inplace = True)

##=============================================================================================##
## Create a Plot of the Data:                                                                  ##
##=============================================================================================##

# Set the title of the plot:

title = "Angular Position vs Time"

# Set the x-axis and y-axis labels:

x_label = "Time (s)"
y_label = "Angle (rad)"

# Create the model, color, and label lists:

model_list = [predictions_df]
color_list = ["red"]
label_list = ["Linearized Model"]

# Create a scatterplot of the feature data vs the target data:

graph = plot_data(X[x_label], y[y_label], title, [x_label, y_label], model_list, color_list,
                  label_list, error_display = True, error = predictions_df['Loss'])

<a name="Learning-Outcomes"></a>

---

#<font size = 6> <b> 3. Create A Simple Neural Network </b> </font>

---

In [ ]:
##=============================================================================================##
## Scale the Feature Data With Standard Scalar and Then Split into Trainging and Testing Sets: ##
##=============================================================================================##

X_norm = ## YOUR CODE HERE

# Split the data into testing and training subsets (test_size = 0.2, random_state = 42):

X_train_norm, X_test_norm, y_train, y_test = ## YOUR CODE HERE


In [ ]:
##=============================================================================================##
## Create A Simple Neural Network With 1 Hidden Neuron (10 Nodes) and Output Neuron (1 Node):  ##
##=============================================================================================##

# Start with a sigmoid activation function.

model_simple_NN = ## YOUR CODE HERE

model_simple_NN.compile(optimizer = 'adam', loss = 'mse')
model_simple_NN.summary()

In [ ]:
##=============================================================================================##
## Fit the Model With the Non-Linearized Training Data (We want it to find the pattern itself):##
##=============================================================================================##

# Start with 10 epochs and batch_size 1, then play around with the values.

## YOUR CODE HERE

In [ ]:
##=============================================================================================##
## Get The Model Predictions:                                                                  ##
##=============================================================================================##

predictions = ## YOUR CODE HERE

##=============================================================================================##
## Calculate Model Losses:                                                                     ##
##=============================================================================================##

loss = ## YOUR CODE HERE

print("Simple NN Model Loss: ", np.round(loss, 5), "rad")

##=============================================================================================##
## Create a DataFrame to Store the Model's Results and Predictions:                            ##
##=============================================================================================##

# Create the results DataFrame:

results_df = pd.DataFrame({"Name": "Linear Regression Model", "Coefs": [coefs], "Bias": bias, "Loss": loss})

# Create the predictions DataFrame:

predictions_df = pd.DataFrame({"Feature": X_test.iloc[:, 0]})

predictions_df["Predictions"] = predictions

predictions_df["Loss"] = loss

predictions_df.sort_index(inplace = True)

##=============================================================================================##
## Create a Plot of the Data:                                                                  ##
##=============================================================================================##

# Set the title of the plot:

title = "Angular Position vs Time"

# Set the x-axis and y-axis labels:

x_label = "Time (s)"
y_label = "Angle (rad)"

# Create the model, color, and label lists:

model_list = [predictions_df]
color_list = ["red"]
label_list = ["Linear Model"]

# Create a scatterplot of the feature data vs the target data:

graph = plot_data(X[x_label], y[y_label], title, [x_label, y_label], model_list, color_list,
                  label_list, error_display = True, error = predictions_df['Loss'])

<a name="Learning-Outcomes"></a>

---

#<font size = 6> <b> 4. Create A More Complex Neural Network </b> </font>

---

In [ ]:
##=============================================================================================##
## Create A Neural Network With 2+ Hidden Neurons (200+ Nodes Each) and Output Neuron (1 Node):##
##=============================================================================================##

# Use 'relu' activation function and play around with NN size and shape.

model_complex_NN = ## YOUR CODE HERE

model_complex_NN.compile(optimizer = 'adam', loss = 'mse')
model_complex_NN.summary()

In [ ]:
##=============================================================================================##
## Fit the Model With the Non-Linearized Training Data (We want it to find the pattern itself):##
##=============================================================================================##

# Play around with epochs and batch_size until you get a good loss.

## YOUR CODE HERE

In [ ]:
##=============================================================================================##
## Get The Model Predictions:                                                                  ##
##=============================================================================================##

predictions = ## YOUR CODE HERE

##=============================================================================================##
## Calculate Model Losses:                                                                     ##
##=============================================================================================##

loss = ## YOUR CODE HERE

print("Complex NN Model Loss: ", np.round(loss, 5), "rad")

##=============================================================================================##
## Create a DataFrame to Store the Model's Results and Predictions:                            ##
##=============================================================================================##

# Create the results DataFrame:

results_df = pd.DataFrame({"Name": "Linear Regression Model", "Coefs": [coefs], "Bias": bias, "Loss": loss})

# Create the predictions DataFrame:

predictions_df = pd.DataFrame({"Feature": X_test.iloc[:, 0]})

predictions_df["Predictions"] = predictions

predictions_df["Loss"] = loss

predictions_df.sort_index(inplace = True)

##=============================================================================================##
## Create a Plot of the Data:                                                                  ##
##=============================================================================================##

# Set the title of the plot:

title = "Angular Position vs Time"

# Set the x-axis and y-axis labels:

x_label = "Time (s)"
y_label = "Angle (rad)"

# Create the model, color, and label lists:

model_list = [predictions_df]
color_list = ["red"]
label_list = ["Linear Model"]

# Create a scatterplot of the feature data vs the target data:

graph = plot_data(X[x_label], y[y_label], title, [x_label, y_label], model_list, color_list,
                  label_list, error_display = True, error = predictions_df['Loss'])

1) Compare and contrast the utility of each of the 4 model types (simple linear, linearized, simple NN, complex NN).

2) Which model was best when (a) you know the underlying physics and (b) You don't know what's happening with the underlying physics.